In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;

Init();

In [ ]:
string proj = "SlipConvergence_Droplet_Zoom";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [3]:
//BoSSSshell.WorkflowMgm.Sessions.ElementAt(5).GetSessionDirectory()

In [4]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination);
sessions.ForEach(s => display(s));

In [ ]:
var sess_90_freeslip = sessions.Where(s => s.Name.Contains("meshStudy_4_AMR16_P4_CA90.00_SLInfinity_ST0.00_HV1,000.00")).Single();
var sess_80_freeslip = sessions.Where(s => s.Name.Contains("meshStudy_4_AMR16_P4_CA80.00_SLInfinity_ST0.00_HV1,000.00")).Single();
var sess_90_noslip = sessions.Where(s => s.Name.Contains("meshStudy_4_AMR16_P4_CA90.00_SL0.00_ST0.00_HV1,000.00")).Single();
var sess_80_noslip = sessions.Where(s => s.Name.Contains("meshStudy_4_AMR16_P4_CA80.00_SL0.00_ST0.00_HV1,000.00")).Single();

In [6]:
// sess_80_freeslip.Timesteps.Last().Export().To(Path.GetFullPath(Path.Combine(".", "SlipConvergenceSpline", sess_80_freeslip.Name))).WithShadowFields().WithSupersampling(2).Do();

In [7]:
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.PrintingNip;

In [8]:
static Func<double, double>[] ExtractSplines(ISessionInfo s){
    var P = s.Timesteps.Last().Fields.Where(f => f.Identification == "Pressure").Single();
    var Ux = s.Timesteps.Last().Fields.Where(f => f.Identification == "VelocityX").Single();
    var Uy = s.Timesteps.Last().Fields.Where(f => f.Identification == "VelocityY").Single();
    var T = s.Timesteps.Last().Fields.Where(f => f.Identification == "Temperature").Single();
    var LS = s.Timesteps.Last().Fields.Where(f => f.Identification == "Phi").Single();

    EdgeMask em = new EdgeMask(LS.Basis.GridDat, X => Math.Abs(X[1]) < 1e-12); // lower wall
    var lSspline = Postprocessing.SplineOnEdge(em, LS, 0, out _, out _);
    var pSpline =new[] {Postprocessing.SplineOnEdge(em, ((XDGField)P).GetSpeciesShadowField("A"), 0, out _, out _), Postprocessing.SplineOnEdge(em, ((XDGField)P).GetSpeciesShadowField("B"), 0, out _, out _)};
    var uxSpline =new[] {Postprocessing.SplineOnEdge(em, ((XDGField)Ux).GetSpeciesShadowField("A"), 0, out _, out _), Postprocessing.SplineOnEdge(em, ((XDGField)Ux).GetSpeciesShadowField("B"), 0, out _, out _)};
    var uySpline =new[] {Postprocessing.SplineOnEdge(em, ((XDGField)Uy).GetSpeciesShadowField("A"), 0, out _, out _), Postprocessing.SplineOnEdge(em, ((XDGField)Uy).GetSpeciesShadowField("B"), 0, out _, out _)};
    var tSpline =new[] {Postprocessing.SplineOnEdge(em, ((XDGField)T).GetSpeciesShadowField("A"), 0, out _, out _), Postprocessing.SplineOnEdge(em, ((XDGField)T).GetSpeciesShadowField("B"), 0, out _, out _)};

    Func<double, double>[] ret = new Func<double, double>[5];
    ret[0] = x => lSspline.Interpolate(x) < 0.0 ? pSpline[0].Interpolate(x) : pSpline[1].Interpolate(x);
    ret[1] = x => lSspline.Interpolate(x) < 0.0 ? uxSpline[0].Interpolate(x) : uxSpline[1].Interpolate(x);
    ret[2] = x => lSspline.Interpolate(x) < 0.0 ? uySpline[0].Interpolate(x) : uySpline[1].Interpolate(x);
    ret[3] = x => lSspline.Interpolate(x) < 0.0 ? tSpline[0].Interpolate(x) : tSpline[1].Interpolate(x);
    ret[4] = x => lSspline.Interpolate(x);


    return ret;
}  

In [9]:
var spline = new[,] {{ExtractSplines(sess_90_noslip), ExtractSplines(sess_90_freeslip)},
                    {ExtractSplines(sess_80_noslip), ExtractSplines(sess_80_freeslip)}}; // 1st index angle, 2nd index slip

In [11]:
void ExportData(string filename, double[] x, double[] y){
    using(var stw = new StringWriter()) {   

        int n = x.Length;
        for(int i = 0; i < n; i++){
            stw.WriteLine($"{x[i].ToString()} {y[i].ToString()}");
        }
        
        Directory.CreateDirectory(BoSSSshell.wmg.CurrentProject);
        File.WriteAllText(Path.Combine(".",BoSSSshell.wmg.CurrentProject, filename), stw.ToString());
    }
    
}

In [27]:
int N = 100000;
var x = GenericBlas.Linspace(-1.5, 1.5, N);
double[,][][] X = new double[2,2][][];
double[,][][] Y = new double[2,2][][];

for(int i = 0; i < 2; i++){
    for(int j = 0; j < 2; j++){
        X[i,j] = new double[3][];
        Y[i,j] = new double[3][];
        for(int k = 0; k < 3; k++){
            X[i,j][k] = x;
            Y[i,j][k] = x.Select(xx => spline[i,j][k](xx)).ToArray();
            // ExportData($"Angle{i}Slip{j}Field{k}.txt", X[i,j][k], Y[i,j][k]);
        }            
    }
}

Plot(x, x.Select(xx => spline[1,0][0](xx)-spline[1,0][0](0.0)).ToArray(), "Pressure").Display();
Plot(x, x.Select(xx => spline[1,0][1](xx)).ToArray(), "VelocityX").Display();
Plot(x, x.Select(xx => spline[1,0][2](xx)).ToArray(), "VelocityY").Display();
// Plot(x, x.Select(xx => spline[1,1][3](xx)).ToArray(), "Temperature").Display();
// Plot(x, x.Select(xx => spline[1,1][4](xx)).ToArray(), "Phi").Display();



In [ ]:
int N = 1000;
double[,][][] X = new double[2,2][][];
double[,][][] Y = new double[2,2][][];
double delta = 0.01;

double a = 0.816;
double b = 0.784;
double theta = 80.0/180.0*Math.PI;
double y0 = -b/Math.Sqrt(1.0+a*a/(b*b)*Math.Tan(theta)*Math.Tan(theta));
double[] x;

for(int i = 1; i < 2; i++){
    for(int j = 0; j < 2; j++){
        
        double x0 = i == 1 ? a*Math.Cos(Math.Asin(y0/b)) : a;
        
        x = GenericBlas.Linspace(-1.5, -x0-delta, N).Cat(GenericBlas.Linspace(-x0-delta, -x0, 5*N)).Cat(GenericBlas.Linspace(-x0, -x0+delta, 5*N)).Cat(GenericBlas.Linspace(-x0+delta, x0-delta, 2*N)).Cat(GenericBlas.Linspace(x0-delta, x0, 5*N)).Cat(GenericBlas.Linspace(x0, x0+delta, 5*N)).Cat(GenericBlas.Linspace(x0+delta, 1.5, N));
        
        X[i,j] = new double[3][];
        Y[i,j] = new double[3][];
        for(int k = 0; k < 3; k++){
            X[i,j][k] = x;
            Y[i,j][k] = x.Select(xx => spline[i,j][k](xx)).ToArray();
            //ExportData($"Angle{i}Slip{j}Field{k}_adaptive.txt", X[i,j][k], Y[i,j][k]);
        }            
    }
}

Plot(x, x.Select(xx => spline[1,0][0](xx)-spline[1,0][0](0.0)).ToArray(), "Pressure").Display();
Plot(x, x.Select(xx => spline[1,0][1](xx)).ToArray(), "VelocityX").Display();
Plot(x, x.Select(xx => spline[1,0][2](xx)).ToArray(), "VelocityY").Display();
// Plot(x, x.Select(xx => spline[1,1][3](xx)).ToArray(), "Temperature").Display();
// Plot(x, x.Select(xx => spline[1,1][4](xx)).ToArray(), "Phi").Display();



In [462]:
using MathNet.Numerics.RootFinding;

In [ ]:
(double[], double[]) GetValuesIn(Func<double, double>[,][] f, int angle, int field, int slip){
    int i = angle; // theta 90, theta 80
    int j = field; // pressure, velocity x, velocity y, temperature
    
    double a = 0.816;
    double b = 0.784;
    double theta = 80.0/180.0*Math.PI;
    double y0 = -b/Math.Sqrt(1.0+a*a/(b*b)*Math.Tan(theta)*Math.Tan(theta));
    double x0_ex = i == 1 ? a*Math.Cos(Math.Asin(y0/b)) : a;
    double x0 = Bisection.FindRoot(t => f[i,slip][4](t), 0, 1.5);
    (Math.Abs(x0-x0_ex)).Display();
    
    var x = GenericBlas.Logspace(5.722/10*1e-6, x0/2, 1000);   // cellsize/10     
    var y = x.Select(xx => Math.Abs(f[i,slip][j](x0-xx)-f[i,slip][j](0))).ToArray();
    return (x,y);
}

int field = 2;
var f1 = GetValuesIn(spline, 0, field, 0);
var f2 = GetValuesIn(spline, 0, field, 1);
var f3 = GetValuesIn(spline, 1, field, 0);
var f4 = GetValuesIn(spline, 1, field, 1);
         
Gnuplot plt = new Gnuplot();
// plt.PlotLogXLogY(x.Select(xx => Math.Abs(xx)), y.Select(yy => Math.Abs(yy-y.Max())));
// plt.PlotLogSlope(-1.5, new[] {1e-9, 1e6});
plt.PlotLogXLogY(f1.Item1, f1.Item2, title : "noslip90", format: new PlotFormat("-k"));
plt.PlotLogXLogY(f2.Item1, f2.Item2, title : "freeslip90", format: new PlotFormat("--k"));
plt.PlotLogXLogY(f3.Item1, f3.Item2, title : "noslip80", format: new PlotFormat(":k"));
plt.PlotLogXLogY(f4.Item1, f4.Item2, title : "freeslip80", format: new PlotFormat("-.k"));

plt.Cmd("set grid");
plt.PlotNow().Display();

double[,][][] X = new double[2,2][][];
double[,][][] Y = new double[2,2][][];

for(int i = 0; i < 2; i++){
    for(int j = 0; j < 2; j++){
        X[i,j] = new double[3][];
        Y[i,j] = new double[3][];
        for(int k = 0; k < 3; k++){
            var vv = GetValuesIn(spline, i,k,j);
            X[i,j][k] = vv.Item1;
            Y[i,j][k] = vv.Item2;
            //ExportData($"LogLogAngle{i}Slip{j}Field{k}Inn.txt", X[i,j][k], Y[i,j][k]);
        }            
    }
}

In [ ]:
(double[], double[]) GetValuesOut(Func<double, double>[,][] f, int angle, int field, int slip){
    int i = angle; // theta 90, theta 80
    int j = field; // pressure, velocity x, velocity y, temperature
    
    double a = 0.816;
    double b = 0.784;
    double theta = 80.0/180.0*Math.PI;
    double y0 = -b/Math.Sqrt(1.0+a*a/(b*b)*Math.Tan(theta)*Math.Tan(theta));
    double x0_ex = i == 1 ? a*Math.Cos(Math.Asin(y0/b)) : a;
    double x0 = Bisection.FindRoot(t => f[i,slip][4](t), 0, 1.5);
    (Math.Abs(x0-x0_ex)).Display();
    
    var x = GenericBlas.Logspace(5.722/10*1e-6, 1.5-x0, 1000);   // cellsize/10     
    var y = x.Select(xx => Math.Abs(f[i,slip][j](x0+xx)-f[i,slip][j](1.5))).ToArray();
    return (x,y);
}

int field = 2;
var f1 = GetValuesOut(spline, 0, field, 0);
var f2 = GetValuesOut(spline, 0, field, 1);
var f3 = GetValuesOut(spline, 1, field, 0);
var f4 = GetValuesOut(spline, 1, field, 1);
         
Gnuplot plt = new Gnuplot();
// plt.PlotLogXLogY(x.Select(xx => Math.Abs(xx)), y.Select(yy => Math.Abs(yy-y.Max())));
// plt.PlotLogSlope(-1.5, new[] {1e-9, 1e6});
plt.PlotLogXLogY(f1.Item1, f1.Item2, title : "noslip90", format: new PlotFormat("-k"));
plt.PlotLogXLogY(f2.Item1, f2.Item2, title : "freeslip90", format: new PlotFormat("--k"));
plt.PlotLogXLogY(f3.Item1, f3.Item2, title : "noslip80", format: new PlotFormat(":k"));
plt.PlotLogXLogY(f4.Item1, f4.Item2, title : "freeslip80", format: new PlotFormat("-.k"));

plt.Cmd("set grid");
plt.PlotNow().Display();

double[,][][] X = new double[2,2][][];
double[,][][] Y = new double[2,2][][];

for(int i = 0; i < 2; i++){
    for(int j = 0; j < 2; j++){
        X[i,j] = new double[3][];
        Y[i,j] = new double[3][];
        for(int k = 0; k < 3; k++){
            var vv = GetValuesOut(spline, i,k,j);
            X[i,j][k] = vv.Item1;
            Y[i,j][k] = vv.Item2;
            //ExportData($"LogLogAngle{i}Slip{j}Field{k}Out.txt", X[i,j][k], Y[i,j][k]);
        }            
    }
}

In [406]:
ilPSP.DoubleExtensions.LogLogRegressionSlope(f4.Item1.Take(700), f4.Item2.Take(700)).Display()

In [433]:
double GetSlope(Func<double, double>[,][] f, int angle, int field, int slip, bool inn){
    int i = angle; // theta 90, theta 80
    int j = field; // pressure, velocity x, velocity y, temperature
    
    double a = 0.816;
    double b = 0.784;
    double theta = 80.0/180.0*Math.PI;
    double y0 = -b/Math.Sqrt(1.0+a*a/(b*b)*Math.Tan(theta)*Math.Tan(theta));
    double x0_ex = i == 1 ? a*Math.Cos(Math.Asin(y0/b)) : a;
    double x0 = Bisection.FindRoot(t => f[i,slip][4](t), 0, 1.5);
    //(Math.Abs(x0-x0_ex)).Display();

    if(inn){
        // var x = GenericBlas.Logspace(5.722/10*1e-6, 5.722/10*1e-2, 1000);   // cellsize/10     
        var x = GenericBlas.Logspace(0.02, 0.2, 1000);   // cellsize/10     
        var y = x.Select(xx => Math.Abs(f[i,slip][j](x0-xx)-f[i,slip][j](0.0))).ToArray();
        return ilPSP.DoubleExtensions.LogLogRegressionSlope(x,y);
    } else{
        // var x = GenericBlas.Logspace(5.722/10*1e-6, 5.722/10*1e-2, 1000);   // cellsize/10     
        var x = GenericBlas.Logspace(0.02, 0.2, 1000);   // cellsize/10     
        var y = x.Select(xx => Math.Abs(f[i,slip][j](x0+xx)-f[i,slip][j](1.5))).ToArray();    
        return ilPSP.DoubleExtensions.LogLogRegressionSlope(x,y);
    }
}

In [ ]:
GetSlope(spline, 0, 0, 0, true).Display();
GetSlope(spline, 0, 0, 1, true).Display();
GetSlope(spline, 1, 0, 0, true).Display();
GetSlope(spline, 1, 0, 1, true).Display();
Console.WriteLine();
GetSlope(spline, 0, 0, 0, false).Display();
GetSlope(spline, 0, 0, 1, false).Display();
GetSlope(spline, 1, 0, 0, false).Display();
GetSlope(spline, 1, 0, 1, false).Display();

In [435]:
GetSlope(spline, 0, 2, 0, true).Display();
